# 🦜 Project 13 — LangChain Document QA

> **Load any text or PDF document and ask natural-language questions about it — using LangChain's document loaders, text splitters, FAISS, and RetrievalQA chain.**

This notebook walks through the core **Retrieval-Augmented Generation (RAG)** pattern step by step:

```
Load document → Split into chunks → Embed chunks → Build FAISS index → Ask questions
```

The sample document covers AWS cloud services — a domain you can swap for any text or PDF of your own.

## ⚙️ Setup

Install dependencies and set your OpenAI API key before running the cells below.

In [ ]:
# Install required packages (run once)
# !pip install langchain>=0.2.0 langchain-openai>=0.1.0 langchain-community>=0.2.0 \
#             langchain-text-splitters>=0.2.0 faiss-cpu>=1.7.4 openai>=1.0.0 tiktoken>=0.5.0

import os

# Set your OpenAI API key here or via environment variable
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-key-here")

print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))

## 📄 Step 1 — Prepare the Document

We embed a multi-section AWS reference document directly in the notebook and write it to disk.
In a real project, replace `doc_path` with any `.txt` file, or use `PyPDFLoader` for PDFs.

In [ ]:
SAMPLE_TEXT = """
Amazon Web Services (AWS) is the world's most comprehensive and broadly adopted cloud platform,
offering over 200 fully featured services from data centres globally.

== Compute ==
Amazon EC2 (Elastic Compute Cloud) provides scalable virtual servers in the cloud. Instance
types include compute-optimised (C-series), memory-optimised (R-series), storage-optimised
(I-series), and GPU instances (P and G series). EC2 Auto Scaling adjusts capacity automatically.

AWS Lambda is a serverless compute service that runs code without provisioning servers.
Lambda scales automatically from a few requests per day to thousands per second.

Amazon ECS (Elastic Container Service) and Amazon EKS (Elastic Kubernetes Service) manage
containerised workloads. ECS is AWS-native; EKS provides managed Kubernetes. AWS Fargate
runs containers on both without managing the underlying EC2 instances.

== Storage ==
Amazon S3 delivers 99.999999999% (eleven 9s) of durability. Storage classes include
S3 Standard, S3 Intelligent-Tiering, S3 Standard-IA, S3 Glacier Instant Retrieval,
and S3 Glacier Deep Archive.

Amazon EBS provides persistent block storage for EC2 instances in SSD (gp3, io2)
and HDD (st1, sc1) volume types.

Amazon EFS is a scalable, elastic NFS file system that grows and shrinks automatically.

== Databases ==
Amazon RDS supports Aurora, PostgreSQL, MySQL, MariaDB, Oracle, and SQL Server.
Multi-AZ deployments provide high availability.

Amazon DynamoDB is a serverless NoSQL database with single-digit millisecond latency at scale.

Amazon Redshift is a petabyte-scale managed data warehouse using standard SQL.

== Infrastructure as Code ==
AWS CloudFormation provisions infrastructure via JSON or YAML templates.

HashiCorp Terraform uses HCL to define, provision, and manage AWS infrastructure declaratively.
Terraform state files track current infrastructure state for incremental updates.
Terraform modules package reusable infrastructure patterns.

Ansible integrates with AWS via the community.aws collection for post-provisioning
configuration management. Packer builds hardened AWS AMIs using Ansible provisioners.

== Migration ==
AWS Migration Hub tracks server and application migrations. AWS Application Migration
Service (MGN) handles lift-and-shift migrations.

AWS DMS (Database Migration Service) migrates databases with the source remaining
operational during migration, minimising downtime.
"""

doc_path = "sample_document.txt"
with open(doc_path, "w", encoding="utf-8") as f:
    f.write(SAMPLE_TEXT.strip())

print(f"Document written: {doc_path} ({len(SAMPLE_TEXT)} characters)")

## ✂️ Step 2 — Load and Split into Chunks

`TextLoader` reads the file into a list of `Document` objects. `RecursiveCharacterTextSplitter`
then breaks each document into overlapping chunks. The overlap (50 chars) ensures context
isn't lost at chunk boundaries — a sentence split across two chunks still gets retrieved together.

**Why chunk?** LLMs have token limits. A 50-page document can't fit in a single prompt, but
the 3-5 most relevant chunks almost always can.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the document
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")
print(f"Full document length: {len(documents[0].page_content)} characters")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # max characters per chunk
    chunk_overlap=50,      # overlap between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],  # try paragraph breaks first
)
chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks")

# Inspect a few chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200], "...")

## 🔢 Step 3 — Embed Chunks and Build FAISS Vector Store

Each chunk is converted into a dense 1536-dimensional embedding vector using OpenAI's
`text-embedding-3-small` model. FAISS then indexes these vectors for ultra-fast nearest-neighbour
similarity search at query time.

When you ask a question, the question is also embedded, and FAISS returns the `k` chunks
whose vectors are most similar (cosine similarity) to the question vector.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("Generating embeddings with text-embedding-3-small...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"FAISS index ready — {vectorstore.index.ntotal} vectors stored")

# Quick similarity search test
test_results = vectorstore.similarity_search("What is Amazon S3?", k=2)
print("\nTop 2 chunks for 'What is Amazon S3?':")
for i, doc in enumerate(test_results):
    print(f"  [{i+1}] {doc.page_content[:150]}...")

## 🔗 Step 4 — Build the RetrievalQA Chain

The `RetrievalQA` chain combines:
- A **retriever** (FAISS similarity search, top-3 chunks)
- An **LLM** (GPT-3.5-turbo at temperature 0 for factual consistency)
- A **custom prompt** that instructs the LLM to answer only from the provided context

Setting `return_source_documents=True` means every answer comes with the exact chunks
that were used — essential for debugging and building trust in the system.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Custom prompt — keeps the LLM grounded in the document
prompt_template = """You are a helpful assistant that answers questions based strictly
on the provided context. If the answer is not contained in the context, say
"I don't know based on the provided document." Do not fabricate information.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",                        # stuff all chunks into one prompt
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt},
)

print("RetrievalQA chain ready!")

## ❓ Step 5 — Ask Questions

Each call to `qa_chain.invoke()` embeds the question, retrieves the top-3 chunks,
and passes them with the question to the LLM. The LLM generates an answer grounded
only in those chunks.

In [ ]:
def ask(question: str):
    """Run a question through the QA chain and display the result."""
    result = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {result['result']}")
    print(f"   └─ Based on {len(result['source_documents'])} chunk(s)\n")
    return result

# Preset questions about AWS
questions = [
    "What is Amazon EC2 and what instance types are available?",
    "How durable is Amazon S3 and what storage classes does it offer?",
    "What is the difference between ECS and EKS?",
    "How does Terraform integrate with AWS?",
    "What AWS services are used for database migration?",
]

for q in questions:
    ask(q)

In [ ]:
# Test the 'I don't know' behaviour — asking about something not in the document
ask("What is the price of an EC2 t3.micro instance?")

In [ ]:
# Inspect source documents for a specific question
result = qa_chain.invoke({"query": "How does Ansible work with AWS?"})
print("Answer:", result["result"])
print("\nSource chunks used:")
for i, doc in enumerate(result["source_documents"]):
    print(f"\n  Chunk {i+1}:")
    print(f"  {doc.page_content}")

## 📝 Summary

In this project you built a complete **document QA system** using LangChain's RAG pattern:

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | `TextLoader` | Reads the file into `Document` objects |
| 2 | `RecursiveCharacterTextSplitter` | Chunks the document with overlap |
| 3 | `OpenAIEmbeddings` + `FAISS` | Embeds chunks, builds searchable index |
| 4 | `RetrievalQA` + `PromptTemplate` | Retrieves relevant chunks, generates answer |

### What to explore next
- **Project 16** (`16-langchain-rag-faiss`) — deeper RAG with multiple documents, metadata filtering, and MMR retrieval
- **Project 19** (`19-langchain-summarisation`) — summarise documents too long to fit in one prompt using map-reduce
- **Project 23** (`23-langchain-chroma-vectordb`) — persist your vector DB to disk with Chroma
- Try replacing `TextLoader` with `PyPDFLoader` to query your own PDFs
- Experiment with different `chunk_size` and `k` values and observe how answer quality changes